# Learning from Demonstrations

<img src="https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/refs/heads/main/assets/logo.jpg?raw=true" style="float: left; width: 15%" />

[CSC_53439_EP-2026](https://moodle.ip-paris.fr/course/view.php?id=19695) Lab session #4

2019-2026 Jérémie Decock

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab4_lfd_and_pbrl.ipynb)

[![My Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main?filepath=lab4_lfd_and_pbrl.ipynb)

[![NbViewer](https://raw.githubusercontent.com/jupyter/design/master/logos/Badges/nbviewer_badge.svg)](https://nbviewer.jupyter.org/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab4_lfd_and_pbrl.ipynb)

[![Local](https://img.shields.io/badge/Local-Save%20As...-blue)](https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/main/lab4_lfd_and_pbrl.ipynb)

## Introduction

The purpose of this lab is to introduce some classic algorithms of *Learning from Demonstrations*. We will see how they work, their caveats and benefits.

*Learning from Demonstrations* (LfD) is an approach in reinforcement learning where an agent learns behaviors by observing examples provided by a demonstrator. This field is divided into two main branches: *Imitation Learning* and *Inverse Reinforcement Learning* (IRL).

- **Imitation Learning** involves directly mimicking the demonstrator's actions. The agent learns to replicate the demonstrated behavior without attempting to infer the underlying objectives or rewards driving the actions. It focuses on reproducing successful behaviors in a given task, often through supervised learning.

- **Inverse Reinforcement Learning (IRL)**, on the other hand, seeks to infer the demonstrator's underlying reward function. Instead of copying actions, the agent tries to discover the goals or preferences that motivated the demonstrator's behavior. Once the reward function is learned, the agent can optimize its own policy to achieve similar outcomes.

The main objective of *Learning from Demonstrations* is to enable agents to learn effectively in settings where the reward function is unknown or poorly defined, by leveraging expert demonstrations as a source of supervisory signal.

Both parts of this lab focus on *Imitation Learning*. In the first part, we will implement the *Behavioral Cloning* algorithm, while the second part (bonus) will introduce the *GAIL* algorithm, which borrows ideas from both Imitation Learning and Inverse Reinforcement Learning:
- Part 1 (Behavioral Cloning):
  - Exercise 1: discover the MountainCar environment and write a handcrafted expert policy;
  - Exercise 2: Behavioral Cloning on MountainCar;
  - Exercise 3: discover the LunarLander environment;
  - Exercise 4: Behavioral Cloning on LunarLander;
- Part 2 (GAIL, bonus):
  - Exercise 5: GAIL on LunarLander.

You can either:
- open, edit and execute the notebook in *Google Colab* following this link: https://colab.research.google.com/github/jeremiedecock/polytechnique-csc-53439-ep-2026-students/blob/main/lab4_lfd_and_pbrl.ipynb ; this is the **recommended** choice as you have nothing to install on your computer
- open, edit and execute the notebook in *MyBinder* (if for any reason the Google Colab solution doesn't work): https://mybinder.org/v2/gh/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main?filepath=lab4_lfd_and_pbrl.ipynb
- download, edit and execute the notebook on your computer if Python3 and JupyterLab are already installed: https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/main/lab4_lfd_and_pbrl.ipynb

If you work with Google Colab or MyBinder, **remember to save or download your work regularly or you may lose it!**

## Lab Submission

Please submit your completed notebook in [Moodle: "Lab 4 - Submission"](https://moodle.ip-paris.fr/mod/assign/view.php?id=432464).

### Submission Guidelines

1. **File Naming:** Rename your notebook as follows: **`firstname_lastname-04.ipynb`** where `firstname` and `lastname` match your email address. *Example: `jesse_read-04.ipynb`*
2. **Clear Output Cells:** To reduce file size (**must be under 500 KB**), clear all output cells before submitting. This includes rendered images, videos, plots, and dataframes...
   - **JupyterLab:**
     - Click **"Kernel" → "Restart Kernel and Clear Outputs of All Cells..."**
     - Then go to **"File" → "Save Notebook As..."**
   - **Google Colab:**
     - Click **"Edit" → "Clear all outputs"**
     - Then go to **"File" → "Download" → "Download .ipynb"**
   - **VSCode:**
     - Click **"Clear All Outputs"**
     - Then **save your file**
3. **Upload Your File:** Only **`.ipynb`** files are accepted.

**Note:** Bonus parts (if any) are optional, as their name suggests.

## Setup the Python environment

This notebook relies on several libraries including `torch`, `gymnasium` (with the `box2d` extra for LunarLander), `numpy`, `pandas`, `seaborn`, `moviepy`, `pygame`, and `tqdm`.
A complete list of dependencies can be found in the following [requirements_lab4.txt](https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab4.txt) file.

### If you use Google Colab

If you use Google Colab, execute the next cell to install required libraries.

In [ ]:
import sys
import subprocess

def is_colab():
    return "google.colab" in sys.modules

def run_subprocess_command(cmd):
    # run the command (stderr is redirected to stdout so that error messages are printed too)
    process = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    # print the output
    for line in process.stdout:
        print(line.decode().strip())
    # check that the command succeeded
    if process.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {process.returncode}): {cmd}")

if is_colab():
    run_subprocess_command("apt-get install -y swig")    # Required to build Box2D (used by the LunarLander environment)
    run_subprocess_command("pip install -r https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab4_google_colab.txt")

### If you have downloaded the notebook on your computer and execute it in your own Python environment

To set up the necessary dependencies, run the following commands to establish a [Python virtual environment (venv)](https://docs.python.org/3/library/venv.html) that includes all the essential libraries for this lab.

#### On POSIX systems (Linux, macOS, WSL, ...)

```bash
python3 -m venv env-lab4
source env-lab4/bin/activate
python3 -m pip install --upgrade pip
python3 -m pip install -r https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab4.txt
```

#### On Windows

```bat
python -m venv env-lab4
env-lab4\Scripts\activate.bat
python -m pip install --upgrade pip
python -m pip install -r https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab4.txt
```

These commands are for the Command Prompt (`cmd.exe`). In PowerShell, activate the virtual environment with `env-lab4\Scripts\Activate.ps1` instead.

#### Install SWIG (required by Box2D)

The LunarLander environment relies on the [Box2D](https://box2d.org/) physics engine. On some systems, the `box2d-py` Python package has to be compiled during its installation, which requires the [SWIG](https://www.swig.org/) tool. If the installation of the requirements fails on `box2d-py`, install SWIG first and try again. For instance:
- on Debian GNU/Linux, Ubuntu, ...: `sudo apt install swig build-essential`
- on macOS (with [Homebrew](https://brew.sh/)): `brew install swig`
- on Windows: `python -m pip install swig` (you may also need the [Microsoft C++ Build Tools](https://visualstudio.microsoft.com/visual-cpp-build-tools/))

### Run CSC-53439-EP notebooks locally in a VSCode Dev Container

If you use [Visual Studio Code](https://code.visualstudio.com/) and have Docker (or Podman) installed on your computer, the course repository includes a ready-to-use [Dev Container](https://code.visualstudio.com/docs/devcontainers/containers) configuration (in the `.devcontainer/` directory) that provides Python 3.12, SWIG and all the libraries required for this lab.

1. Install the [Dev Containers](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-containers) extension in VSCode.
   If you use Podman instead of Docker, set the `dev.containers.dockerPath` setting to `podman` in VSCode.
2. Clone the repository and open it in VSCode:
   ```bash
   git clone https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students.git
   code polytechnique-csc-53439-ep-2026-students
   ```
3. When VSCode asks, click **"Reopen in Container"** (or open the Command Palette with `Ctrl+Shift+P` / `Cmd+Shift+P` and run **"Dev Containers: Reopen in Container"**).
   The first start may take several minutes, as the image is built and the Python dependencies listed in `requirements.txt` are installed automatically.
4. Open this notebook and select the container's Python interpreter as the kernel: **"Select Kernel" → "Python Environments..." → "Python 3.12"**.

The repository folder on your computer is shared with the container, so your work is kept when the container is stopped or rebuilt.

### Import required packages

In [ ]:
import json
import lzma
from pathlib import Path
from typing import Callable, List, Optional, Tuple
import urllib.request

import gymnasium as gym
import numpy as np
import pandas as pd
import torch

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

import seaborn as sns
from tqdm.notebook import tqdm

In [ ]:
from IPython.display import Video

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
gym.__version__

In [ ]:
sns.set_context("talk")

In [ ]:
FIGS_DIR = Path("figs/") / "lab4"       # Where to save figures (.mp4 files)
PLOTS_DIR = Path("figs/") / "lab4"      # Where to save plots (.png or .svg files)
MODELS_DIR = Path("models/")            # Where the expert demonstrations are stored

In [ ]:
FIGS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## PyTorch setup

PyTorch can run on both CPUs and GPUs. The following cell will determine the device PyTorch will use. If a GPU is available, PyTorch will use it; otherwise, it will use the CPU.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Set the device to CUDA if available, otherwise use CPU

For utilizing a GPU on Google Colab, you also have to activate it following the steps outlined [here](https://colab.research.google.com/notebooks/gpu.ipynb).

In [ ]:
print("Available GPUs:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"- Device {i}: {torch.cuda.get_device_name(i)}")
else:
    print("- No GPU available.")

If you have a recent GPU and want to use it, you may need to install a specific version of PyTorch compatible with your CUDA version. For this, you will have to edit the [requirements_lab4.txt](https://raw.githubusercontent.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/main/requirements_lab4.txt) file and replace the current version of PyTorch with one compatible with your CUDA version. Check the [official PyTorch website](https://pytorch.org/get-started/locally/) for more information.

Note that a GPU is not very useful in this lab because the neural networks and the datasets are small, and CUDA spends more time transferring data between the CPU and GPU than processing it directly on the CPU.

You can uncomment the next cell to explicitly instruct PyTorch to train neural networks using the CPU.

In [ ]:
# device = "cpu"

In [ ]:
print(f"PyTorch will train and test neural networks on {device}")

## Helper functions

In this lab, a *policy* is simply a Python function that takes an observation (a NumPy array) as input and returns an action (an integer).

The two functions defined in the next cell will be used in all the exercises:
- `record_episode` runs one episode of an environment with a given policy, records it as an MP4 video (in `FIGS_DIR`) and returns this video to display it in the notebook;
- `evaluate_policy` runs several episodes of an environment with a given policy and returns the return (sum of rewards) of each episode.

In [ ]:
def record_episode(env_id: str, policy: Callable[[np.ndarray], int], video_dirname: str, seed: Optional[int] = None) -> Video:
    """
    Run one episode of the environment `env_id` with `policy`, record it as an MP4 video and return this video.

    Parameters
    ----------
    env_id : str
        The Gymnasium identifier of the environment (e.g. "MountainCar-v0").
    policy : Callable[[np.ndarray], int]
        The policy: a function that takes an observation as input and returns an action.
    video_dirname : str
        The name of the directory (in `FIGS_DIR`) where the video is saved.
    seed : int, optional
        The seed used to reset the environment (None for a random initial state).

    Returns
    -------
    Video
        The video of the episode (displayed in the notebook when it is returned by the last line of a cell).
    """
    env = gym.make(env_id, render_mode="rgb_array")
    env = gym.wrappers.RecordVideo(env, video_folder=str(FIGS_DIR / video_dirname))

    observation, info = env.reset(seed=seed)
    episode_return = 0.0
    num_steps = 0
    terminated = truncated = False

    # Run the episode until it ends (terminal state or time limit)
    while not (terminated or truncated):
        action = policy(observation)
        observation, reward, terminated, truncated, info = env.step(action)
        episode_return += float(reward)
        num_steps += 1

    env.close()

    print(f"Episode return: {episode_return:.1f} ({num_steps} steps)")

    return Video(FIGS_DIR / video_dirname / "rl-video-episode-0.mp4", embed=True, html_attributes="controls autoplay loop")


def evaluate_policy(env_id: str, policy: Callable[[np.ndarray], int], num_episodes: int = 20, seed: int = 0) -> np.ndarray:
    """
    Run `num_episodes` episodes of the environment `env_id` with `policy` and return the return of each episode.

    Parameters
    ----------
    env_id : str
        The Gymnasium identifier of the environment (e.g. "MountainCar-v0").
    policy : Callable[[np.ndarray], int]
        The policy: a function that takes an observation as input and returns an action.
    num_episodes : int
        The number of episodes to run.
    seed : int
        The seed used to reset the environment at the first episode (episode `i` uses the seed `seed + i`).

    Returns
    -------
    np.ndarray
        The return (sum of rewards) of each episode.
    """
    env = gym.make(env_id)
    episode_return_list = []

    for episode_index in range(num_episodes):
        observation, info = env.reset(seed=seed + episode_index)
        episode_return = 0.0
        terminated = truncated = False

        while not (terminated or truncated):
            action = policy(observation)
            observation, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)

        episode_return_list.append(episode_return)

    env.close()

    return np.array(episode_return_list)

## Part 1: Behavioral Cloning

### Exercise 1: Hands on the MountainCar environment

MountainCar is a classic reinforcement learning environment. In this simple 2D scenario, an underpowered car must navigate a hill, but it lacks the power to ascend directly. Instead, the car must learn to use the hill's slopes to build momentum and ultimately reach the flag at the top. While the environment is straightforward, it becomes interesting due to its sparse reward signal (a reward of -1 is given at each time step, until the flag is reached or the episode is truncated after 200 steps), making it an excellent candidate for learning from demonstrations.

**Task 1.1:** Refer to the following link [MountainCar Environment](https://gymnasium.farama.org/environments/classic_control/mountain_car/) to familiarize yourself with the MountainCar environment if you are not already.

Print some information about the environment:

In [ ]:
env = gym.make("MountainCar-v0", render_mode="rgb_array")

mountain_car_state_dim = env.observation_space.shape[0]
mountain_car_action_dim = int(env.action_space.n)

print(f"State space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print("Actions are: {" + ", ".join([str(a) for a in range(env.action_space.n)]) + "}")

env.close()

The observation is the pair (position of the car, velocity of the car), and the 3 actions are: 0 = accelerate to the left, 1 = don't accelerate, 2 = accelerate to the right.

**Task 1.2:** Run the following cells and check different basic policies (for instance constant actions or randomly drawn actions) to discover the MountainCar environment.

#### Test the MountainCar environment with a constant policy

**Practical tips:** the next cells use Python *lambda expressions* to define very simple policies in a single line.
A lambda expression is just a compact way to define a small function without giving it a name: `lambda observation: 0` is a function that takes an observation as input and always returns the action 0 (whatever the observation).

Thus, the following line:

```python
record_episode("MountainCar-v0", lambda observation: 0, "lab4_mountain-car_action0")
```

is equivalent to:

```python
def policy(observation):
    return 0

record_episode("MountainCar-v0", policy, "lab4_mountain-car_action0")
```

More generally, `lambda x: <expression>` defines the same function as `def f(x): return <expression>`.
Lambda expressions are also used below to define the random policy, and in Exercise 3.

In [ ]:
record_episode("MountainCar-v0", lambda observation: 0, "lab4_mountain-car_action0")

In [ ]:
record_episode("MountainCar-v0", lambda observation: 1, "lab4_mountain-car_action1")

In [ ]:
record_episode("MountainCar-v0", lambda observation: 2, "lab4_mountain-car_action2")

#### Test the MountainCar environment with a random policy

In [ ]:
mountain_car_action_space = gym.make("MountainCar-v0").action_space

record_episode("MountainCar-v0", lambda observation: int(mountain_car_action_space.sample()), "lab4_mountain-car_random_action")

#### Test the MountainCar environment with a good handcrafted policy

**Task 1.3:** The MountainCar environment is simple in design but poses a significant challenge for many algorithms, such as PPO, due to its sparse reward structure. The agent must engage in extensive exploration before receiving its first informative reward, which occurs only when it successfully reaches the flag at the top of the hill. Despite this, the task can be solved with a surprisingly simple policy. Can you discover it?

Implement it in the `mountain_car_handcrafted_policy` function below. This policy will be our *expert* in the next exercise.

In [ ]:
def mountain_car_handcrafted_policy(observation: np.ndarray) -> int:
    """
    A handcrafted policy for the MountainCar environment.

    Parameters
    ----------
    observation : np.ndarray
        The observation (position, velocity) of the car.

    Returns
    -------
    int
        The action to take (0 = accelerate to the left, 1 = don't accelerate, 2 = accelerate to the right).
    """
    position, velocity = observation

    # action = TODO... # Choose the action to take from the position and the velocity of the car

    return action

In [ ]:
record_episode("MountainCar-v0", mountain_car_handcrafted_policy, "lab4_mountain-car_handcrafted_policy")

A good policy reaches the flag in every episode, i.e. in less than 200 steps (the return of an episode is then greater than -200).
Execute the next cell to evaluate your policy on 20 episodes.

In [ ]:
mountain_car_handcrafted_policy_returns = evaluate_policy("MountainCar-v0", mountain_car_handcrafted_policy, num_episodes=20)

print(f"Handcrafted policy: mean return = {mountain_car_handcrafted_policy_returns.mean():.1f} ± {mountain_car_handcrafted_policy_returns.std():.1f}")
print(f"The flag is reached in {(mountain_car_handcrafted_policy_returns > -200).sum()} episodes out of {len(mountain_car_handcrafted_policy_returns)}")

**Answer:**

TODO...

### Behavioral Cloning

*Behavioral Cloning* ([D. A. Pomerleau, *Efficient Training of Artificial Neural Networks for Autonomous Navigation*, Neural Computation, vol. 3, no. 1, pp. 88–97, 1991](https://cours.etsmtl.ca/sys843/REFS/ORG/pomerleau_alvinn.pdf)) is one of the most fundamental approaches to *Imitation Learning*. The concept is straightforward: an *expert* provides high-quality traces, or demonstrations, and the learning agent's task is to mimic the expert's behavior.
In *Imitation Learning*, *traces* or *demonstrations* refer to sequences of state-action pairs generated by an expert while performing a task. These demonstrations serve as examples for the agent to learn from. Each demonstration consists of a series of observations (states) encountered by the expert, along with the corresponding actions taken in those states.
For example, in a driving task, a demonstration might be a series of snapshots of the environment (such as the car's position and speed) and the actions the expert driver took at each moment (such as steering or braking). These state-action pairs are recorded and used to train the agent, enabling it to learn how to behave similarly in similar situations.

The goal of *Behavioral Cloning* is to map states to the actions the expert would take, essentially allowing the agent to "clone" the expert's behavior. The quality and variety of these demonstrations are critical for successful learning, as they provide the agent with the knowledge it needs to act appropriately across different scenarios.
Typically, expert demonstrations are obtained by recording human behavior, which is then used to train the agent.

In other words, Behavioral Cloning turns imitation into a *supervised learning* problem: the inputs are the states visited by the expert and the labels are the actions chosen by the expert.
With a discrete action space, this is a classification problem: in this lab, the policy network outputs one score (logit) per action and is trained by minimizing the cross-entropy between its predictions and the expert's actions.
This is equivalent to maximizing the likelihood of the expert's actions under the stochastic policy defined by the softmax of these logits.
When the trained policy is used to act in the environment, we simply choose the action with the highest score (i.e. we use a deterministic, greedy policy).

### Exercise 2: Behavioral Cloning on MountainCar

#### Make a PyTorch dataset from the demonstrations

##### Download expert demonstrations

The expert demonstrations are available in the `models/` directory of the course repository.
The `download_demonstrations` function defined in the next cell downloads them in the `MODELS_DIR` directory (unless they are already there, e.g. if you have cloned the repository).

The MountainCar demonstrations have been recorded with the handcrafted policy of Exercise 1: https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/refs/heads/main/models/lab4_expert_mountaincar-v0_handcrafted/demonstrations.json.xz

In [ ]:
DEMONSTRATIONS_BASE_URL = "https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/refs/heads/main/models"

def download_demonstrations(expert_name: str) -> Path:
    """
    Download the demonstrations of an expert (unless they have already been downloaded) and return the path of the downloaded file.

    Parameters
    ----------
    expert_name : str
        The name of the expert (e.g. "lab4_expert_mountaincar-v0_handcrafted").

    Returns
    -------
    Path
        The path of the (compressed JSON) demonstrations file.
    """
    file_path = MODELS_DIR / expert_name / "demonstrations.json.xz"

    if not file_path.exists():
        file_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{DEMONSTRATIONS_BASE_URL}/{expert_name}/demonstrations.json.xz"
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, file_path)

    return file_path

In [ ]:
mountain_car_demonstrations_path = download_demonstrations("lab4_expert_mountaincar-v0_handcrafted")

##### Make the dataset

The demonstrations file contains a list of transitions; each transition is a dictionary with the following keys: `episode`, `t`, `observation`, `action`, `reward`, `next_observation`, `terminated` and `truncated`.

The `ExpertDemonstrationsDataset` class defined in the next cell is a [PyTorch dataset](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) made from these demonstrations: its items are the (observation, action) pairs of all the transitions.

In [ ]:
class ExpertDemonstrationsDataset(torch.utils.data.Dataset):
    """
    A PyTorch dataset of (observation, action) pairs made from expert demonstrations.

    Attributes
    ----------
    observations : torch.Tensor
        The observations of all the transitions, shape (num_transitions, observation_dim).
    actions : torch.Tensor
        The actions of all the transitions, shape (num_transitions,).
    episode_returns : np.ndarray
        The return (sum of rewards) of each demonstrated episode.
    """

    def __init__(self, json_demonstrations_file_path: Path, transform=None, target_transform=None):
        super().__init__()
        self.json_demonstrations_file_path = json_demonstrations_file_path

        with lzma.open(self.json_demonstrations_file_path, "rt") as f:
            demonstrations_list = json.load(f)

        self.observations = torch.tensor([transition["observation"] for transition in demonstrations_list], dtype=torch.float32)
        self.actions = torch.tensor([transition["action"] for transition in demonstrations_list], dtype=torch.long)

        # Compute the return of each episode (to compare the expert with the trained agents)
        episode_returns_dict = {}
        for transition in demonstrations_list:
            episode_returns_dict[transition["episode"]] = episode_returns_dict.get(transition["episode"], 0.) + transition["reward"]
        self.episode_returns = np.array(list(episode_returns_dict.values()))

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self) -> int:
        return len(self.observations)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        observation = self.observations[idx]
        action = self.actions[idx]

        if self.transform:
            observation = self.transform(observation)
        if self.target_transform:
            action = self.target_transform(action)

        return observation, action

In [ ]:
mountain_car_expert_dataset = ExpertDemonstrationsDataset(mountain_car_demonstrations_path)

**Task 2.1:** Take time to check the definition and the content of this dataset.

In [ ]:
print(f"Number of transitions: {len(mountain_car_expert_dataset)}")
print(f"Number of episodes: {len(mountain_car_expert_dataset.episode_returns)}")
print(f"Mean return of the demonstrated episodes: {mountain_car_expert_dataset.episode_returns.mean():.1f}")
print(f"First item (observation, action): {mountain_car_expert_dataset[0]}")

##### Plot the dataset

The next cell plots the action chosen by the expert in each observation (position, velocity) of the dataset.

In [ ]:
def plot_mountain_car_policy(observations: torch.Tensor, actions: torch.Tensor, title: str):
    """
    Plot the action chosen in each observation (position, velocity) of the MountainCar environment.
    """
    policy_df = pd.DataFrame({
        "position": observations[:, 0].cpu().numpy(),
        "velocity": observations[:, 1].cpu().numpy(),
        "action": actions.cpu().numpy()
    })
    ax = policy_df.plot(kind="scatter", x="position", y="velocity", c="action", colormap="viridis", colorbar=True, figsize=(10, 7), s=2)
    ax.set_title(title)
    plt.show()

In [ ]:
plot_mountain_car_policy(mountain_car_expert_dataset.observations, mountain_car_expert_dataset.actions, "Expert policy (demonstrations)")

#### Define the neural network

**Task 2.2:** Implement a neural network that takes the state as input and outputs one score (logit) per action.

The architecture of the network is up to you: what is the simplest neural network able to imitate the expert's policy?
Hint: look at the shape of the expert's *decision boundary* in the observation space (position, velocity), i.e. the boundary between the regions where the expert chooses different actions (see your handcrafted policy of Task 1.3 and the plot of the dataset above).
If you choose a network with hidden layers, define their size directly in the class.
Justify your choice in the answer cell below.

In [ ]:
class MountainCarPolicyNetwork(torch.nn.Module):

    def __init__(self, n_observations: int, n_actions: int):

        super(MountainCarPolicyNetwork, self).__init__()

        # TODO... # Define the layer(s) of the network

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # logits = TODO... # Compute the score (logit) of each action

        return logits

**Answer:**

TODO...

In [ ]:
mountain_car_model = MountainCarPolicyNetwork(n_observations=mountain_car_state_dim, n_actions=mountain_car_action_dim).to(device)
print(mountain_car_model)

#### Define the training loop

**Task 2.3:** Implement a classical supervised learning training function that trains the neural network for one epoch on the dataset (i.e. one pass over all the batches of the dataloader). This function will be called at each epoch in the *Train the model* cell defined below.

In [ ]:
def train(dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_fn: Callable, optimizer: torch.optim.Optimizer, verbose: bool = True) -> None:

    dataset_size = len(dataloader.dataset)

    # TODO... # Set the model in training mode

    # for TODO... # Iterate over the batches (X, y_true) of the dataloader
    #     X, y_true = TODO... # Send the batch to `device`

    #     y_pred = TODO... # Compute the predictions of the model
    #     train_loss = TODO... # Compute the prediction error with `loss_fn`

    #     TODO... # Backpropagation (zero_grad, backward, step)

    #     TODO... # If `verbose` is True, print the training loss every 100 batches

#### Define the testing loop

**Task 2.4:** Implement a classical supervised learning testing function that assesses the performance of the neural network on the test dataset: it has to return the average loss and the accuracy (i.e. the percentage of observations for which the predicted action, the one with the highest score, is the expert's action). This function will be called at each epoch in the *Train the model* cell defined below.

In [ ]:
def test(dataloader: torch.utils.data.DataLoader, model: torch.nn.Module, loss_fn: Callable) -> Tuple[float, float]:

    # TODO... # Set the model in evaluation mode

    # TODO... # In a `torch.no_grad()` block, iterate over the batches (X, y_true) of the dataloader: send them to `device`, compute the predictions of the model and accumulate the loss and the number of correct predictions (the predicted action is the one with the highest score)

    # test_loss = TODO... # Compute the average loss over the batches
    # test_accuracy = TODO... # Compute the accuracy (in %) over the whole dataset

    return test_loss, test_accuracy

#### Split the dataset

In the following cell, we split the dataset (expert demonstrations) into a training set (80% of the transitions) and a test set (20% of the transitions).

In [ ]:
mountain_car_train_subset, mountain_car_test_subset = torch.utils.data.random_split(mountain_car_expert_dataset, [0.8, 0.2])

mountain_car_train_dataloader = torch.utils.data.DataLoader(mountain_car_train_subset, batch_size=32, shuffle=True)
mountain_car_test_dataloader = torch.utils.data.DataLoader(mountain_car_test_subset, batch_size=32, shuffle=False)

#### Define the loss function and the optimizer

**Task 2.5:** Define the loss function used to train the neural network.

In [ ]:
# mountain_car_loss_fn = TODO... # Define the loss function (see https://docs.pytorch.org/docs/stable/nn.html#loss-functions)

mountain_car_optimizer = torch.optim.Adam(mountain_car_model.parameters(), lr=0.1)

**Note:** the learning rate used here (0.1) is much higher than the default learning rate of Adam (0.001).
The velocity of the car is very small (between -0.07 and 0.07): for the decision boundary of the linear model expected in Task 2.2 to be close to the line `velocity = 0` whatever the position of the car, the weights applied to the velocity must be much larger than its other parameters (they reach several hundreds during the training).
As Adam changes each parameter by at most about the learning rate at each step, reaching such weights would take far too many epochs with the default learning rate (you can try it).

#### Train the model

In [ ]:
epochs = 50
for epoch in range(epochs):
    train(mountain_car_train_dataloader, mountain_car_model, mountain_car_loss_fn, mountain_car_optimizer, verbose=False)
    test_loss, test_accuracy = test(mountain_car_test_dataloader, mountain_car_model, mountain_car_loss_fn)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:>3d}/{epochs}: test accuracy = {test_accuracy:>5.1f}%, test loss = {test_loss:>8f}")

With the handcrafted expert, you should reach a test accuracy of about 99% (the few errors are made very close to the line `velocity = 0`, where the expert's action changes).

#### Make a policy from the trained model

The `make_greedy_policy` function defined in the next cell turns a trained policy network into a policy (a function that takes an observation and returns an action), that can be used with the `record_episode` and `evaluate_policy` functions: it chooses the action with the highest score.

In [ ]:
def make_greedy_policy(model: torch.nn.Module) -> Callable[[np.ndarray], int]:
    """
    Make a greedy policy from a policy network: the chosen action is the one with the highest output of the network.
    """
    model_device = next(model.parameters()).device

    def policy(observation: np.ndarray) -> int:
        with torch.no_grad():
            observation_tensor = torch.tensor(observation, dtype=torch.float32, device=model_device)
            return int(model(observation_tensor).argmax().item())

    return policy

In [ ]:
mountain_car_bc_policy = make_greedy_policy(mountain_car_model)

##### Plot the learned policy

You can now compare the learned policy with the expert's policy, on the observations of the test set...

In [ ]:
mountain_car_test_observations = mountain_car_expert_dataset.observations[mountain_car_test_subset.indices]

mountain_car_model.eval()
with torch.no_grad():
    mountain_car_predicted_actions = mountain_car_model(mountain_car_test_observations.to(device)).argmax(dim=1)

plot_mountain_car_policy(mountain_car_test_observations, mountain_car_predicted_actions, "Learned policy (test set observations)")

... and on the whole observation space, including the observations that are never visited by the expert (and for which the policy network has never been trained).

In [ ]:
positions, velocities = torch.meshgrid(torch.linspace(-1.2, 0.6, 100), torch.linspace(-0.07, 0.07, 100), indexing="ij")
mountain_car_grid_observations = torch.stack([positions.flatten(), velocities.flatten()], dim=1)

with torch.no_grad():
    mountain_car_grid_predicted_actions = mountain_car_model(mountain_car_grid_observations.to(device)).argmax(dim=1)

plot_mountain_car_policy(mountain_car_grid_observations, mountain_car_grid_predicted_actions, "Learned policy (whole observation space)")

Compare this plot with the rule of the expert (accelerate to the right if and only if `velocity >= 0`): with the linear model expected in Task 2.2, the learned decision boundary is a straight line in the whole observation space, including in the regions that are not covered by the demonstrations.
This line is very close to the expert's one (`velocity = 0`), thus the learned policy extrapolates the expert's rule far beyond the demonstrations: the *inductive bias* of the model (linear decision boundaries) matches the structure of the expert's policy.
A network with hidden layers can fit the demonstrations as well, but nothing constrains its decision boundary to be a straight line in the regions that are not covered by the demonstrations.

#### Test the learned policy

In [ ]:
record_episode("MountainCar-v0", mountain_car_bc_policy, "lab4_mountain-car_bc_policy")

In [ ]:
mountain_car_bc_policy_returns = evaluate_policy("MountainCar-v0", mountain_car_bc_policy, num_episodes=20)

print(f"Expert (handcrafted) policy: mean return = {mountain_car_handcrafted_policy_returns.mean():.1f} ± {mountain_car_handcrafted_policy_returns.std():.1f}")
print(f"Behavioral Cloning policy:   mean return = {mountain_car_bc_policy_returns.mean():.1f} ± {mountain_car_bc_policy_returns.std():.1f}")
print(f"The flag is reached in {(mountain_car_bc_policy_returns > -200).sum()} episodes out of {len(mountain_car_bc_policy_returns)}")

Here, the Behavioral Cloning policy should be almost as good as the expert and reach the flag in every episode: as the (linear) policy network has the same kind of decision boundary as the expert, it can imitate the expert almost perfectly, in the whole observation space.
This is not always the case: with a more complex expert, the Behavioral Cloning policy may be significantly worse than the expert despite a high test accuracy (see Task 4.3).

### Exercise 3: Hands on the LunarLander environment

In this exercise, we will apply the *Behavioral Cloning* algorithm to the LunarLander environment. The LunarLander environment is a classic reinforcement learning task where an agent must learn to land a spacecraft safely on a landing pad. The agent controls the spacecraft's engines and has four possible actions: do nothing, fire the left orientation engine, fire the main engine, or fire the right orientation engine.
The agent is rewarded for moving towards the landing pad and for landing safely on it (+100), and it is penalized for moving away from it, for crashing (-100) and each time it fires an engine. An episode is considered a solution if it scores at least 200 points.

The reward signal is less sparse than in the MountainCar environment, but the dynamics are more complex, making it an interesting environment for *Imitation Learning* too.

**Task 3.1:** Refer to the following link [LunarLander-v3 Environment](https://gymnasium.farama.org/environments/box2d/lunar_lander/) to familiarize yourself with the LunarLander-v3 environment if you are not already.

Print some information about the environment:

In [ ]:
env = gym.make("LunarLander-v3", render_mode="rgb_array")

lunar_lander_state_dim = env.observation_space.shape[0]
lunar_lander_action_dim = int(env.action_space.n)

print(f"State space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print("Actions are: {" + ", ".join([str(a) for a in range(env.action_space.n)]) + "}")

env.close()

**Task 3.2:** Run the following cells and check different basic policies (for instance constant actions or randomly drawn actions) to discover the LunarLander-v3 environment.

#### Test the LunarLander environment with a constant policy

In [ ]:
record_episode("LunarLander-v3", lambda observation: 0, "lab4_lunar-lander_action0")

In [ ]:
record_episode("LunarLander-v3", lambda observation: 1, "lab4_lunar-lander_action1")

In [ ]:
record_episode("LunarLander-v3", lambda observation: 2, "lab4_lunar-lander_action2")

In [ ]:
record_episode("LunarLander-v3", lambda observation: 3, "lab4_lunar-lander_action3")

#### Test the LunarLander environment with a random policy

In [ ]:
lunar_lander_action_space = gym.make("LunarLander-v3").action_space

record_episode("LunarLander-v3", lambda observation: int(lunar_lander_action_space.sample()), "lab4_lunar-lander_random_action")

### Exercise 4: Behavioral Cloning on LunarLander

In this exercise, we will reuse most of the code of Exercise 2 (the `ExpertDemonstrationsDataset` class and the `train`, `test` and `make_greedy_policy` functions) to apply the *Behavioral Cloning* algorithm to the LunarLander environment.
The main change is the policy network: unlike the handcrafted MountainCar expert, the LunarLander expert is a neural network trained with PPO, whose decision boundaries are not linear; a more expressive model than a linear one is thus needed to imitate it.

#### Make a PyTorch dataset from the demonstrations

##### Download expert demonstrations

The LunarLander demonstrations have been recorded with an expert policy trained with PPO (only the episodes with a return of at least 250 have been kept): https://github.com/jeremiedecock/polytechnique-csc-53439-ep-2026-students/raw/refs/heads/main/models/lab4_expert_lunar-lander-v2-discrete-nowind_ppo/demonstrations.json.xz

In [ ]:
lunar_lander_demonstrations_path = download_demonstrations("lab4_expert_lunar-lander-v2-discrete-nowind_ppo")

In [ ]:
lunar_lander_expert_dataset = ExpertDemonstrationsDataset(lunar_lander_demonstrations_path)

print(f"Number of transitions: {len(lunar_lander_expert_dataset)}")
print(f"Number of episodes: {len(lunar_lander_expert_dataset.episode_returns)}")
print(f"Mean return of the demonstrated episodes: {lunar_lander_expert_dataset.episode_returns.mean():.1f}")

#### Define the neural network

**Task 4.1:** Implement a neural network that takes the state as input and outputs one score (logit) per action. Unlike the linear model expected in Task 2.2, this neural network has a hidden layer, which allows it to learn non-linear decision boundaries. It should have the following architecture:
- A first fully connected layer with `hidden_units` units and ReLU activation function
- A second fully connected layer with `n_actions` units and no activation function

A linear model is not expressive enough to imitate the LunarLander expert accurately: with the training settings used below, a linear model (like the one expected in Task 2.2) only reaches a test accuracy of about 75% on this dataset (you can try it), against about 90% for a network with a hidden layer.

In [ ]:
class DiscretePolicyNetwork(torch.nn.Module):

    def __init__(self, n_observations: int, n_actions: int, hidden_units: int):

        super(DiscretePolicyNetwork, self).__init__()

        # self.layer1 = TODO...
        # self.layer2 = TODO...

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        # x = TODO... # Apply `layer1` and the ReLU activation function
        # logits = TODO... # Apply `layer2` (no activation function)

        return logits

In [ ]:
lunar_lander_model = DiscretePolicyNetwork(n_observations=lunar_lander_state_dim, n_actions=lunar_lander_action_dim, hidden_units=64).to(device)
print(lunar_lander_model)

#### Split the dataset

In [ ]:
lunar_lander_train_subset, lunar_lander_test_subset = torch.utils.data.random_split(lunar_lander_expert_dataset, [0.8, 0.2])

lunar_lander_train_dataloader = torch.utils.data.DataLoader(lunar_lander_train_subset, batch_size=32, shuffle=True)
lunar_lander_test_dataloader = torch.utils.data.DataLoader(lunar_lander_test_subset, batch_size=32, shuffle=False)

#### Define the loss function and the optimizer

**Task 4.2:** Define the loss function used to train the neural network.

In [ ]:
# lunar_lander_loss_fn = TODO... # Define the loss function (see https://docs.pytorch.org/docs/stable/nn.html#loss-functions)

lunar_lander_optimizer = torch.optim.Adam(lunar_lander_model.parameters())

#### Train the model

In [ ]:
epochs = 10
for epoch in range(epochs):
    train(lunar_lander_train_dataloader, lunar_lander_model, lunar_lander_loss_fn, lunar_lander_optimizer, verbose=False)
    test_loss, test_accuracy = test(lunar_lander_test_dataloader, lunar_lander_model, lunar_lander_loss_fn)

    print(f"Epoch {epoch + 1:>3d}/{epochs}: test accuracy = {test_accuracy:>5.1f}%, test loss = {test_loss:>8f}")

#### Test the learned policy

In [ ]:
lunar_lander_bc_policy = make_greedy_policy(lunar_lander_model)

record_episode("LunarLander-v3", lunar_lander_bc_policy, "lab4_lunar-lander_bc_policy")

In [ ]:
lunar_lander_bc_policy_returns = evaluate_policy("LunarLander-v3", lunar_lander_bc_policy, num_episodes=20)

print(f"Expert policy (demonstrations): mean return = {lunar_lander_expert_dataset.episode_returns.mean():.1f} ± {lunar_lander_expert_dataset.episode_returns.std():.1f}")
print(f"Behavioral Cloning policy:      mean return = {lunar_lander_bc_policy_returns.mean():.1f} ± {lunar_lander_bc_policy_returns.std():.1f}")
print(f"Number of episodes with a return of at least 200: {(lunar_lander_bc_policy_returns >= 200).sum()} out of {len(lunar_lander_bc_policy_returns)}")

**Task 4.3:** Compare the returns obtained by the Behavioral Cloning policy with the returns of the expert. The test accuracy of the policy network is high (around 90%): why can the Behavioral Cloning policy still perform significantly worse than the expert (e.g. crash or hover above the landing pad in some episodes)? What could be done to mitigate this problem?

**Answer:**

TODO...

## Part 2: GAIL (bonus part)

### Introduction to GAIL

**Generative Adversarial Imitation Learning (GAIL)** ([Ho & Ermon, 2016](https://proceedings.neurips.cc/paper_files/paper/2016/file/cc7e2b878868cbae992d1fb743995d8f-Paper.pdf)) is an advanced imitation learning algorithm that combines ideas from Generative Adversarial Networks (GANs) and Inverse Reinforcement Learning (IRL).

Unlike Behavioral Cloning, which directly learns to map states to actions through supervised learning, GAIL trains two neural networks simultaneously, and interacts with the environment (without using its reward):

1. **Discriminator**: A neural network $D_{\boldsymbol{w}}(s, a)$ with parameters $\boldsymbol{w}$ that tries to distinguish between state-action pairs from expert demonstrations (generated by the expert policy $\pi_E$) and those generated by the current policy (in this lab, $D_{\boldsymbol{w}}(s, a)$ is the probability that $(s, a)$ comes from the expert);
2. **Policy (Generator)**: A neural network $\pi_{\boldsymbol{\theta}}(a \mid s)$ with parameters $\boldsymbol{\theta}$ that tries to generate trajectories that fool the discriminator into thinking they come from the expert.

The key insight is that the discriminator provides a learned reward signal to train the policy using reinforcement learning (typically TRPO or PPO): the more the discriminator believes that a state-action pair comes from the expert, the higher the reward. This approach addresses some limitations of Behavioral Cloning:
- As the policy is trained on the states it visits itself, it learns to come back to the states visited by the expert: it is much less affected by the distribution shift between training and test states (see Task 4.3);
- It doesn't require explicit reward functions;
- It typically needs far fewer expert demonstrations than Behavioral Cloning (but many more interactions with the environment).

### Exercise 5: GAIL on LunarLander

**Note**: This implementation of GAIL is intentionally kept simple to focus on understanding the core algorithm without the complexity of advanced policy training procedures. We use the **REINFORCE** algorithm (vanilla policy gradient, see Lab 2) to train the policy, which makes the learning process transparent and easier to follow.

The policy and the discriminator are small neural networks, and the policy is called at each time step of the environment to collect trajectories: in this exercise, all computations are done on the CPU (which is faster than the GPU in this case).

#### Define the policy network

**Task 5.1:** Implement a neural network that takes the state as input and outputs the probability of each action. The neural network should have the following architecture:
- Input Layer:
  - The network takes an input with a dimension of `observations_dim`.
- Hidden Layers:
  - The first hidden layer is a fully connected (Linear) layer with 128 units, followed by a ReLU activation function.
  - The second hidden layer is a fully connected (Linear) layer with 128 units, followed by a ReLU activation function.
- Output Layer:
  - The output layer is a fully connected (Linear) layer with `actions_dim` units.
  - This is followed by a Softmax activation function, which ensures that the output is a probability distribution over actions.

In [ ]:
class PolicyNetwork(torch.nn.Module):
    """Policy network for REINFORCE training in GAIL"""

    def __init__(self, observations_dim: int, actions_dim: int):
        super(PolicyNetwork, self).__init__()

        # self.layer1 = TODO...
        # self.layer2 = TODO...
        # self.layer3 = TODO...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the policy network.

        Parameters
        ----------
        x : torch.Tensor
            State tensor of shape (observations_dim) or (batch_size, observations_dim)

        Returns
        -------
        torch.Tensor
            Action probabilities, shape (actions_dim) or (batch_size, actions_dim)
        """
        # x = TODO... # Apply `layer1` and the ReLU activation function
        # x = TODO... # Apply `layer2` and the ReLU activation function
        # action_probabilities = TODO... # Apply `layer3` and the softmax function (over the last dimension)

        return action_probabilities

#### Define the discriminator neural network

**Task 5.2:** Implement a neural network that takes the state-action pair as input and outputs the probability that the pair comes from the expert policy. The neural network should have the following architecture:
- Input Layer:
  - The network takes a concatenated input of observations and actions (one-hot encoded) with a combined dimension of `observations_dim + actions_dim`.
- Hidden Layers:
  - The first hidden layer is a fully connected (Linear) layer with 128 units, followed by a ReLU activation function.
  - The second hidden layer is a fully connected (Linear) layer with 128 units, followed by a ReLU activation function.
- Output Layer:
  - The output layer is a fully connected (Linear) layer with 1 unit.
  - This is followed by a Sigmoid activation function, which outputs a probability indicating whether the input is from the expert or the generated policy.

In [ ]:
class Discriminator(torch.nn.Module):
    """Discriminator network that distinguishes expert from policy trajectories"""

    def __init__(self, observations_dim: int, actions_dim: int):
        super(Discriminator, self).__init__()

        # self.layer1 = TODO...
        # self.layer2 = TODO...
        # self.layer3 = TODO...

    def forward(self, observations: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the discriminator.

        Parameters
        ----------
        observations : torch.Tensor
            Observations tensor of shape (batch_size, observations_dim)
        actions : torch.Tensor
            One-hot encoded actions tensor of shape (batch_size, actions_dim)

        Returns
        -------
        torch.Tensor
            Probability that each (observation, action) pair comes from the expert, shape (batch_size, 1)
        """
        # x = TODO... # Concatenate the observations and the actions (along the last dimension)
        # x = TODO... # Apply `layer1` and the ReLU activation function
        # x = TODO... # Apply `layer2` and the ReLU activation function
        # prob = TODO... # Apply `layer3` and the sigmoid function

        return prob

#### Environment initialization

In [ ]:
env = gym.make("LunarLander-v3")

observations_dim = env.observation_space.shape[0]   # Dimension of the observation space
actions_dim = int(env.action_space.n)               # Number of possible actions

#### Initialize the policy and the discriminator networks and their optimizers

In [ ]:
# Initialization of the networks (on the CPU)
policy = PolicyNetwork(observations_dim, actions_dim)
discriminator = Discriminator(observations_dim, actions_dim)

# Optimizers
policy_optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-3)

#### Expert demonstrations

We reuse the LunarLander expert demonstrations of Exercise 4: `expert_observations` and `expert_actions` contain all the (observation, action) pairs of the demonstrations (the actions are one-hot encoded, as expected by the discriminator).

In [ ]:
expert_observations = lunar_lander_expert_dataset.observations
expert_actions = torch.nn.functional.one_hot(lunar_lander_expert_dataset.actions, num_classes=actions_dim).float()

#### Function to collect agent trajectories

The following function collects the agent's trajectories using the given policy.

It also returns the return of each episode computed with the rewards of the environment: these returns are **not** used to train the agent (GAIL does not use the rewards of the environment), but only to monitor the training.

In [ ]:
def collect_agent_trajectories(
    policy: PolicyNetwork,
    env: gym.Env,
    num_episodes: int = 10
) -> List[Tuple[np.ndarray, np.ndarray, float]]:
    """
    Collect trajectories from the agent using the given policy.

    Parameters
    ----------
    policy : PolicyNetwork
        The neural network policy that outputs action probabilities
    env : gym.Env
        The gymnasium environment
    num_episodes : int, optional
        Number of episodes to collect (default: 10)

    Returns
    -------
    trajectories : list of tuple
        List of tuples (observation_array, action_array, episode_return) for each episode, where:
        - observation_array : numpy.ndarray
            Observations collected during the episode, shape (episode_length, observations_dim)
        - action_array : numpy.ndarray
            Actions taken during the episode, shape (episode_length,)
        - episode_return : float
            The return of the episode computed with the rewards of the environment (only used to monitor the training)
    """
    trajectories = []

    # Collect multiple episodes
    for _ in range(num_episodes):
        observation_list = []
        action_list = []
        episode_return = 0.0

        # Reset the environment to start a new episode
        observation, _ = env.reset()
        terminated = truncated = False

        # Run one episode until it ends (terminal state or time limit)
        while not (terminated or truncated):
            # Convert observation to tensor for the policy network
            observation_tensor = torch.tensor(observation, dtype=torch.float32)

            # Get action probabilities from the policy (no gradient needed for inference)
            with torch.no_grad():
                action_probabilities = policy(observation_tensor)

            # Sample an action from the probability distribution
            action = torch.multinomial(action_probabilities, 1).item()

            # Store the observation and action
            observation_list.append(observation)
            action_list.append(action)

            # Take a step in the environment
            observation, reward, terminated, truncated, info = env.step(action)
            episode_return += float(reward)

        # Store the complete trajectory for this episode
        trajectories.append((np.array(observation_list), np.array(action_list), episode_return))

    return trajectories

#### The main training loop

**Task 5.3:** Implement the main training loop of the Generative Adversarial Imitation Learning (GAIL) algorithm. At each iteration:

1. **Collecting Agent Trajectories** (already implemented):
   - Agent trajectories are collected with the current policy by running `num_agent_episodes` episodes in the environment.

2. **Processing Trajectories for Policy Gradient** (for each trajectory):
   - Observations and actions are converted to tensors, and actions are converted to one-hot encoding.
   - The discriminator's output $D_{\boldsymbol{w}}(s_t, a_t)$ is used as the reward $r_t$ of the agent (this is a simple choice; another common choice is $r_t = -\ln(1 - D_{\boldsymbol{w}}(s_t, a_t))$). The parameters $\boldsymbol{w}$ are kept fixed here: the reward is a fixed signal for the policy update, and no gradient flows from the policy loss to the discriminator.
   - Discounted returns $G_t = r_t + \gamma G_{t+1}$ are calculated using the discount factor `gamma`.

3. **Concatenation of Data**:
   - All observations, actions, and returns from the trajectories are concatenated into single tensors.
   - Returns are normalized (zero mean, unit variance) to reduce the variance of the policy gradient.

4. **Policy Update** (REINFORCE):
   - Log probabilities of the selected actions are computed.
   - The policy loss $\mathcal{L}(\boldsymbol{\theta}) = -\frac{1}{N} \sum_t G_t \ln \pi_{\boldsymbol{\theta}}(a_t \mid s_t)$, where $N$ is the number of agent transitions, is calculated and backpropagated to update the parameters $\boldsymbol{\theta}$ of the policy network.

5. **Data Preparation for the Discriminator**:
   - A batch of expert data, **with as many transitions as the agent data**, is drawn at random from `expert_observations` and `expert_actions` (with balanced classes, the discriminator cannot reach a low loss by simply predicting "agent" everywhere).
   - Observations and actions from both agent and expert are concatenated.
   - Labels are created (0 for agent data, 1 for expert data).

6. **Training the Discriminator**:
   - The discriminator is trained to distinguish between agent and expert data.
   - The discriminator loss is the binary cross-entropy
     $$\mathcal{L}(\boldsymbol{w}) = -\frac{1}{2N} \sum_{i=1}^{N} \Big[ \ln D_{\boldsymbol{w}}(s^E_i, a^E_i) + \ln\big(1 - D_{\boldsymbol{w}}(s_i, a_i)\big) \Big],$$
     where $(s_i, a_i)$ are the $N$ agent transitions and $(s^E_i, a^E_i)$ the $N$ expert transitions of the batch; it is backpropagated to update the parameters $\boldsymbol{w}$ of the discriminator network.

7. **Logging** (already implemented):
   - The losses, the mean output of the discriminator on agent and expert data, and the mean return of the collected episodes (computed with the rewards of the environment) are recorded, and printed every 25 iterations.

This loop iteratively improves the policy by making it more similar to the expert's behavior while simultaneously training the discriminator to better distinguish between agent and expert actions.

The training takes a few minutes on a laptop.

**Note on the discriminator convention:** In this lab, we follow the convention of GANs ([Goodfellow et al., 2014](https://arxiv.org/abs/1406.2661)): the expert data play the role of the "real" data (label 1) and the agent data play the role of the "generated" data (label 0), so $D(s, a)$ is the probability that $(s, a)$ comes from the expert. Be careful if you read the GAIL paper ([Ho & Ermon, 2016](https://proceedings.neurips.cc/paper_files/paper/2016/file/cc7e2b878868cbae992d1fb743995d8f-Paper.pdf)): it uses the opposite convention, i.e. $D(s, a)$ is trained to output 1 on the agent data and 0 on the expert data, and the policy is trained (with TRPO) to minimize the cost $\ln D(s, a)$.

The two conventions are equivalent: swapping the labels simply amounts to replacing $D$ by $1 - D$ (the binary cross-entropy loss is unchanged). Minimizing the cost $\ln D(s, a)$ of the paper is thus the same as maximizing the reward $-\ln(1 - D(s, a))$ with our convention (the "other common choice" mentioned in step 2 above). We use the GAN convention here because REINFORCE (Lab 2) maximizes a return: as $D(s, a)$ is the probability of "being the expert", the output of the discriminator can directly be used as a reward ($r_t = D(s_t, a_t)$).

In [ ]:
num_iterations = 1000
num_agent_episodes = 5
gamma = 0.99  # Discount factor

discriminator_loss_fn = torch.nn.BCELoss()

# Track training metrics
training_history = {
    "iteration": [],
    "discriminator_loss": [],
    "policy_loss": [],
    "discriminator_output_agent": [],
    "discriminator_output_expert": [],
    "env_return": [],
    "episode_length": [],
}

for iteration in tqdm(range(num_iterations)):

    # Collect agent trajectories
    trajectories = collect_agent_trajectories(policy, env, num_episodes=num_agent_episodes)

    all_observations = []
    all_actions = []
    all_returns = []

    # ==================================
    # STEP 1: Process agent trajectories
    # ==================================

    # for observation_array, action_array, _ in trajectories: # Iterate over the collected trajectories (the return computed with the environment rewards is not used for training)
    #     observation_tensor = TODO... # Convert the observations of the trajectory to a tensor of shape (episode_length, observations_dim)
    #     TODO... # Append it to `all_observations`

    #     action_tensor_onehot = TODO... # Convert the actions of the trajectory to a one-hot encoded tensor of shape (episode_length, actions_dim) (see `torch.nn.functional.one_hot`)
    #     TODO... # Append it to `all_actions`

    #     rewards = TODO... # Compute the rewards r_t = D_w(s_t, a_t) with the discriminator, in a `torch.no_grad()` block (tensor of shape (episode_length,))
    #     returns = TODO... # Compute the discounted returns G_t = r_t + gamma * G_{t+1} (tensor of shape (episode_length,))
    #     TODO... # Append them to `all_returns`

    # ======================================
    # STEP 2: Prepare data for policy update
    # ======================================

    # all_observations = TODO... # Concatenate the observations of all the trajectories (tensor of shape (num_agent_transitions, observations_dim))
    # all_actions = TODO... # Concatenate the one-hot encoded actions of all the trajectories (tensor of shape (num_agent_transitions, actions_dim))
    # all_returns = TODO... # Concatenate the returns of all the trajectories (tensor of shape (num_agent_transitions,))

    # all_returns = TODO... # Normalize the returns (zero mean, unit variance; add 1e-8 to the standard deviation to avoid a division by zero)

    # =================================
    # STEP 3: Update the policy network
    # =================================

    # log_prob_actions_tensor = TODO... # Compute the log probabilities log π_θ(a_t|s_t) of the actions taken (tensor of shape (num_agent_transitions,)); add 1e-8 to the probabilities to avoid log(0)

    # policy_loss = TODO... # Compute the policy loss -1/N sum_t G_t log π_θ(a_t|s_t)

    # TODO... # Update the policy network (zero_grad, backward, step)

    # =============================================
    # STEP 4: Prepare data for discriminator update
    # =============================================

    # expert_indices = TODO... # Draw at random the indices of as many expert transitions as agent transitions (see `torch.randint`)
    # expert_observations_batch = TODO... # Get the corresponding observations in `expert_observations`
    # expert_actions_batch = TODO... # Get the corresponding (one-hot encoded) actions in `expert_actions`

    # discriminator_input_observations = TODO... # Concatenate the agent and expert observations
    # discriminator_input_actions = TODO... # Concatenate the agent and expert actions
    # discriminator_labels = TODO... # Create the labels: 0 for agent data, 1 for expert data

    # ========================================
    # STEP 5: Update the discriminator network
    # ========================================

    # predictions = TODO... # Compute the output of the discriminator on the agent and expert data (tensor of shape (2 * num_agent_transitions,))
    # discriminator_loss = TODO... # Compute the binary cross-entropy loss with `discriminator_loss_fn`

    # TODO... # Update the discriminator network (zero_grad, backward, step)

    # Track metrics
    with torch.no_grad():
        discriminator_output_agent = discriminator(all_observations, all_actions).mean().item()
        expert_indices = torch.randint(len(expert_observations), (1000,))
        discriminator_output_expert = discriminator(expert_observations[expert_indices], expert_actions[expert_indices]).mean().item()

    training_history["iteration"].append(iteration)
    training_history["discriminator_loss"].append(discriminator_loss.item())
    training_history["policy_loss"].append(policy_loss.item())
    training_history["discriminator_output_agent"].append(discriminator_output_agent)
    training_history["discriminator_output_expert"].append(discriminator_output_expert)
    training_history["env_return"].append(np.mean([episode_return for _, _, episode_return in trajectories]))
    training_history["episode_length"].append(np.mean([len(action_array) for _, action_array, _ in trajectories]))

    if iteration % 25 == 0:
        print(f"Iteration {iteration:>4d}: discriminator loss = {discriminator_loss.item():.3f}, "
              f"D(agent) = {discriminator_output_agent:.2f}, D(expert) = {discriminator_output_expert:.2f}, "
              f"mean environment return = {training_history['env_return'][-1]:.1f}")

#### Plot training metrics

In [ ]:
training_history_df = pd.DataFrame(training_history)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# Discriminator loss
axes[0, 0].plot(training_history_df["iteration"], training_history_df["discriminator_loss"])
axes[0, 0].set_xlabel("Iteration")
axes[0, 0].set_ylabel("Discriminator loss")
axes[0, 0].set_title("Discriminator loss")
axes[0, 0].grid(True)

# Discriminator outputs
axes[0, 1].plot(training_history_df["iteration"], training_history_df["discriminator_output_agent"], label="Agent data")
axes[0, 1].plot(training_history_df["iteration"], training_history_df["discriminator_output_expert"], label="Expert data")
axes[0, 1].set_xlabel("Iteration")
axes[0, 1].set_ylabel("Mean D(s, a)")
axes[0, 1].set_title("Mean output of the discriminator")
axes[0, 1].legend()
axes[0, 1].grid(True)

# Return of the collected episodes (computed with the rewards of the environment, not used for training)
axes[1, 0].plot(training_history_df["iteration"], training_history_df["env_return"], alpha=0.3)
axes[1, 0].plot(training_history_df["iteration"], training_history_df["env_return"].rolling(20).mean(), label="Moving average (20 iterations)")
axes[1, 0].axhline(lunar_lander_expert_dataset.episode_returns.mean(), color="red", linestyle="--", label="Expert")
axes[1, 0].set_xlabel("Iteration")
axes[1, 0].set_ylabel("Mean return")
axes[1, 0].set_title("Return of the agent's episodes (environment rewards)")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Episode length
axes[1, 1].plot(training_history_df["iteration"], training_history_df["episode_length"], alpha=0.3)
axes[1, 1].plot(training_history_df["iteration"], training_history_df["episode_length"].rolling(20).mean())
axes[1, 1].set_xlabel("Iteration")
axes[1, 1].set_ylabel("Mean episode length")
axes[1, 1].set_title("Length of the agent's episodes")
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "lab4_gail_training_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

#### Test the learned policy

In [ ]:
gail_policy = make_greedy_policy(policy)

record_episode("LunarLander-v3", gail_policy, "lab4_lunar-lander_gail_policy")

In [ ]:
gail_policy_returns = evaluate_policy("LunarLander-v3", gail_policy, num_episodes=20)

print(f"Expert policy (demonstrations): mean return = {lunar_lander_expert_dataset.episode_returns.mean():.1f} ± {lunar_lander_expert_dataset.episode_returns.std():.1f}")
print(f"Behavioral Cloning policy:      mean return = {lunar_lander_bc_policy_returns.mean():.1f} ± {lunar_lander_bc_policy_returns.std():.1f}")
print(f"GAIL policy:                    mean return = {gail_policy_returns.mean():.1f} ± {gail_policy_returns.std():.1f}")

### Conclusion

This implementation of GAIL is intentionally kept simple to focus on understanding the core algorithm without the complexity of advanced policy training procedures. We use the **REINFORCE** algorithm (vanilla policy gradient) to train the policy, which makes the learning process transparent and easier to follow.

However, this simplicity comes at a cost: the training is **much less efficient and less stable** than with more sophisticated approaches, and with the computational budget of this exercise, the GAIL policy usually remains far from the expert (and often below the Behavioral Cloning policy). In the original GAIL paper ([Ho & Ermon, 2016](https://proceedings.neurips.cc/paper_files/paper/2016/file/cc7e2b878868cbae992d1fb743995d8f-Paper.pdf)), the authors used **Trust Region Policy Optimization (TRPO)** to train the policy, which provides better stability and sample efficiency through constrained optimization.

In modern practice, **Proximal Policy Optimization (PPO)** has become the most widely used algorithm for training the policy in GAIL implementations. PPO offers a good balance between:
- Implementation simplicity (compared to TRPO)
- Training stability (compared to REINFORCE)
- Sample efficiency
- Robustness across different environments

For production-level implementations or more challenging environments, consider replacing the REINFORCE-based policy training with PPO or TRPO to achieve significantly better performance and stability. A common trick is also to initialize the GAIL policy with Behavioral Cloning, so that the agent starts from a policy already close to the expert.

Note also that the choice of the reward computed from the discriminator matters. With $r_t = D_{\boldsymbol{w}}(s_t, a_t)$ (or $r_t = -\ln(1 - D_{\boldsymbol{w}}(s_t, a_t))$), the reward is always positive: the longer the episode, the higher the return, whatever the agent does. This *survival bias* encourages the agent to avoid ending the episode, e.g. by hovering above the ground instead of landing (see the plot of the length of the agent's episodes, which often grows towards the time limit of 1000 steps). Conversely, an always negative reward like $r_t = \ln D_{\boldsymbol{w}}(s_t, a_t)$ encourages the agent to end the episode as soon as possible (e.g. by crashing). In environments where the episode length matters, $r_t = \ln D_{\boldsymbol{w}}(s_t, a_t) - \ln(1 - D_{\boldsymbol{w}}(s_t, a_t))$ (used e.g. in AIRL, Fu et al., 2018) is a more neutral choice.

## Bibliography

- Pomerleau, Dean A. "Efficient training of artificial neural networks for autonomous navigation." Neural Computation 3, no. 1 (1991): 88-97. https://cours.etsmtl.ca/sys843/REFS/ORG/pomerleau_alvinn.pdf

- Ross, Stéphane, and Drew Bagnell. "Efficient reductions for imitation learning." In Proceedings of the thirteenth international conference on artificial intelligence and statistics (AISTATS), pp. 661-668. JMLR Workshop and Conference Proceedings, 2010. https://proceedings.mlr.press/v9/ross10a/ross10a.pdf

- Ross, Stéphane, Geoffrey Gordon, and Drew Bagnell. "A reduction of imitation learning and structured prediction to no-regret online learning." In Proceedings of the fourteenth international conference on artificial intelligence and statistics (AISTATS), pp. 627-635. JMLR Workshop and Conference Proceedings, 2011. https://proceedings.mlr.press/v15/ross11a/ross11a.pdf

- Ho, Jonathan, and Stefano Ermon. "Generative adversarial imitation learning." Advances in neural information processing systems 29 (2016). https://proceedings.neurips.cc/paper_files/paper/2016/file/cc7e2b878868cbae992d1fb743995d8f-Paper.pdf

- Fu, Justin, Katie Luo, and Sergey Levine. "Learning robust rewards with adversarial inverse reinforcement learning." In International Conference on Learning Representations (ICLR), 2018. https://arxiv.org/abs/1710.11248

- Laskey, Michael, Jonathan Lee, Roy Fox, Anca Dragan, and Ken Goldberg. "DART: Noise injection for robust imitation learning." In Conference on robot learning (CoRL), pp. 143-156. PMLR, 2017. https://proceedings.mlr.press/v78/laskey17a/laskey17a.pdf

- Osa, Takayuki, Joni Pajarinen, Gerhard Neumann, J. Andrew Bagnell, Pieter Abbeel, and Jan Peters. "An algorithmic perspective on imitation learning." Foundations and Trends in Robotics 7, no. 1-2 (2018): 1-179. https://arxiv.org/abs/1811.06711